# Lab 01 · Faça a reta aprender
**Machine Learning na Prática · Dia 1 · Módulo 02: Regressão Linear**

Os dados de alunos já chegam **limpos**, do jeito que você deixou no Lab 00. Agora o foco é o modelo:

| Etapa | O que fazer |
|---|---|
| 1. Olhar | carregar os dados prontos e visualizar horas × nota |
| 2. Ajustar | descida do gradiente com NumPy, acompanhar a MSE, comparar com `LinearRegression` |
| 3. Avaliar | prever alunos novos; MAE, MSE e R² no teste; regressão múltipla |
| 4. Taxa de aprendizado | convergência, lentidão e divergência |

Procure as células marcadas com **`# TODO`** e substitua os `___`.

### Bibliotecas
No **Google Colab** tudo já vem instalado: esta célula só confere. Em outro ambiente (Jupyter no seu PC), ela instala o que faltar.

In [ ]:
import importlib.util, subprocess, sys
pacotes = {'numpy': 'numpy', 'pandas': 'pandas', 'matplotlib': 'matplotlib', 'sklearn': 'scikit-learn'}
faltando = [pip for modulo, pip in pacotes.items() if importlib.util.find_spec(modulo) is None]
if faltando:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *faltando])
print('Bibliotecas prontas!' if not faltando else f'Instalado agora: {", ".join(faltando)}')

## 0 · Preparação (célula pronta)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PolynomialFeatures
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

plt.rcParams['figure.figsize'] = (8, 4)

### Os dados limpos do Lab 00
A célula abaixo gera os mesmos dados e aplica a **mesma limpeza** do Lab 00 (duplicados, turno, outliers, alvo faltante).
Se você ainda tem o arquivo `alunos_limpo.csv` do Lab 00 nesta sessão, ele é usado direto.

In [ ]:
def gerar_dados(n=400, seed=42):
    rng = np.random.default_rng(seed)
    horas = np.clip(rng.normal(6, 2.2, n), 0.5, 14).round(1)
    faltas = rng.poisson(3, n)
    turno = rng.choice(['manhã', 'tarde', 'noite'], n, p=[.45, .3, .25])
    monitoria = rng.choice(['sim', 'não'], n, p=[.35, .65])
    idade = rng.integers(17, 31, n)
    renda = np.round(rng.lognormal(8.2, 0.5, n), -1)          # R$ por mês, escala grande
    nota = (2.0 + 0.45 * horas - 0.25 * faltas + 0.6 * (monitoria == 'sim')
            - 0.4 * (turno == 'noite') + rng.normal(0, 0.9, n))
    nota = np.clip(nota, 0, 10).round(1)
    df = pd.DataFrame({'horas_estudo': horas, 'faltas': faltas, 'turno': turno,
                       'monitoria': monitoria, 'idade': idade, 'renda_familiar': renda,
                       'nota_final': nota})
    df['situacao'] = np.where(df['nota_final'] >= 6, 'aprovado', 'reprovado')

    # ---- problemas do mundo real ----
    sujo = rng.choice(n, 40, replace=False)
    df.loc[sujo[:14], 'turno'] = rng.choice(['Manhã', 'MANHÃ ', 'manha', 'Noite', ' tarde'], 14)
    df.loc[rng.choice(n, 20, replace=False), 'horas_estudo'] = np.nan
    df.loc[rng.choice(n, 15, replace=False), 'faltas'] = np.nan
    df.loc[rng.choice(n, 8, replace=False), 'nota_final'] = np.nan
    outl = rng.choice(n, 4, replace=False)
    df.loc[outl, 'horas_estudo'] = df.loc[outl, 'horas_estudo'].fillna(7) * 10   # vírgula esquecida
    df = pd.concat([df, df.sample(12, random_state=seed)], ignore_index=True)   # duplicados
    return df.sample(frac=1, random_state=seed).reset_index(drop=True)

def limpar(df):
    """A mesma limpeza do Lab 00."""
    df = df.drop_duplicates().copy()
    df['turno'] = df['turno'].str.strip().str.lower().replace({'manha': 'manhã'})
    df.loc[df['horas_estudo'] > 24, 'horas_estudo'] = np.nan
    return df.dropna(subset=['nota_final']).reset_index(drop=True)

import os
df = pd.read_csv('alunos_limpo.csv') if os.path.exists('alunos_limpo.csv') else limpar(gerar_dados())
print(df.shape)
df.head()

## 1 · Uma feature, uma reta
Começamos com o caso mais simples: prever `nota_final` só com `horas_estudo`.
Para esta parte, deixamos de fora as linhas sem horas (nas próximas etapas o `Pipeline` cuida delas).

In [ ]:
d = df.dropna(subset=['horas_estudo'])
x_tr, x_te, y_tr, y_te = train_test_split(d['horas_estudo'].to_numpy(), d['nota_final'].to_numpy(),
                                          test_size=0.2, random_state=42)

# TODO: gráfico de dispersão horas × nota (treino)
plt.scatter(___, ___, alpha=.5)
plt.xlabel('horas de estudo'); plt.ylabel('nota final'); plt.show()

## 2 · Descida do gradiente com NumPy
O modelo é $\hat y = w\,x + b$ e a perda é a MSE: $L = \frac{1}{n}\sum (y - \hat y)^2$.
As derivadas são:

$$\frac{\partial L}{\partial w} = -\frac{2}{n}\sum x\,(y-\hat y) \qquad \frac{\partial L}{\partial b} = -\frac{2}{n}\sum (y-\hat y)$$

e a atualização é $w \leftarrow w - \alpha \, \frac{\partial L}{\partial w}$ (o mesmo para $b$).

In [ ]:
def descida_gradiente(x, y, alpha=0.02, epocas=3000):
    w, b = 0.0, 0.0
    historico = []
    for _ in range(epocas):
        # TODO: previsão da reta
        y_hat = ___
        erro = y - y_hat
        historico.append((erro ** 2).mean())          # MSE desta época
        # TODO: gradientes da MSE
        dw = ___
        db = ___
        # TODO: passo contra o gradiente
        w = ___
        b = ___
    return w, b, historico

w, b, hist = descida_gradiente(x_tr, y_tr)
print(f'w = {w:.3f} | b = {b:.3f} | MSE final = {hist[-1]:.3f}')

plt.plot(hist); plt.yscale('log')
plt.xlabel('época'); plt.ylabel('MSE (treino)'); plt.title('A perda caindo'); plt.show()

Agora compare com a solução exata do Scikit-learn (mínimos quadrados):

In [ ]:
# TODO: ajuste uma LinearRegression (ela espera X em 2D: x_tr.reshape(-1, 1))
reg = LinearRegression().___(x_tr.reshape(-1, 1), y_tr)
print(f'Scikit-learn: w = {reg.coef_[0]:.3f} | b = {reg.intercept_:.3f}')
print(f'NumPy (GD):   w = {w:.3f} | b = {b:.3f}')

xs = np.linspace(0, 15, 100)
plt.scatter(x_tr, y_tr, alpha=.4)
plt.plot(xs, w * xs + b, 'C1', lw=3, label='descida do gradiente')
plt.plot(xs, reg.predict(xs.reshape(-1, 1)), 'k--', label='LinearRegression')
plt.legend(); plt.xlabel('horas'); plt.ylabel('nota'); plt.show()

**Curiosidade:** os dados foram gerados com uma regra escondida em que cada hora de estudo vale **+0,45** ponto.
O seu `w` chegou perto? Por que não é exatamente 0,45? (Dica: as outras colunas também mexem na nota.)

## 3 · Prever e avaliar
### 3.1 Alunos novos

In [ ]:
novos = np.array([2, 6, 10])
# TODO: use o modelo do Scikit-learn para prever as notas
for h, nota in zip(novos, reg.___(novos.reshape(-1, 1))):
    print(f'{h:>2} h de estudo → nota prevista {nota:.2f}')

### 3.2 Métricas no teste
- **MAE**: erro médio na unidade original (pontos de nota)
- **MSE**: média dos erros ao quadrado (pune erros grandes)
- **R²**: quanto melhor que sempre chutar a média (1 = perfeito, 0 = igual à média, pode ser negativo)

In [ ]:
pred = reg.predict(x_te.reshape(-1, 1))
# TODO: calcule as três métricas no TESTE
print(f'MAE = {___(y_te, pred):.2f} ponto')
print(f'MSE = {___(y_te, pred):.2f}')
print(f'R²  = {___(y_te, pred):.2f}')

### 3.3 Regressão múltipla: todas as features
Agora usamos **todas** as colunas, com o preparo do Lab 00 já montado (célula pronta).
O modelo passa a ser $\hat y = w_1 x_1 + w_2 x_2 + \dots + b$. Comparamos pela **validação cruzada no treino**; o teste fica para o fim.

In [ ]:
X = df.drop(columns=['nota_final', 'situacao'])
y = df['nota_final']
num, cat = ['horas_estudo', 'faltas', 'idade', 'renda_familiar'], ['turno', 'monitoria']
X_tr, X_te, Y_tr, Y_te = train_test_split(X, y, test_size=0.2, random_state=42)

preparo = ColumnTransformer([
    ('num', Pipeline([('imputar', SimpleImputer(strategy='median')), ('padronizar', StandardScaler())]), num),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat),
])
multipla = Pipeline([('preparo', preparo), ('regressao', LinearRegression())])

In [ ]:
# TODO: MAE da validação cruzada (5 folds) só no treino
# (o Scikit-learn devolve o MAE com sinal negativo — por isso o "-" na frente)
mae_cv = -cross_val_score(multipla, X_tr, Y_tr, cv=___, scoring=___)
print(f'MAE na validação cruzada: {mae_cv.mean():.2f} ± {mae_cv.std():.2f}')

# TODO: treine no treino inteiro e avalie UMA vez no teste
multipla.fit(___, ___)
pred = multipla.predict(X_te)
print(f'Teste → MAE = {mean_absolute_error(Y_te, pred):.2f} | R² = {r2_score(Y_te, pred):.2f}')

In [ ]:
coefs = pd.Series(multipla.named_steps['regressao'].coef_,
                  index=multipla.named_steps['preparo'].get_feature_names_out()).round(2)
coefs.sort_values()

**Para pensar:** quais coeficientes são maiores em módulo? Isso bate com a correlação que você viu no Lab 00? Coeficiente prova causa?

## 4 · A taxa de aprendizado
Rode a descida do gradiente com taxas diferentes e compare as curvas de perda.

In [ ]:
# TODO: compare 4 taxas: 0.001 (pequena), 0.01 (boa), 0.023 (quase no limite) e 0.03 (grande demais)
for alpha in [___, ___, ___, ___]:
    _, _, h = descida_gradiente(x_tr, y_tr, alpha=alpha, epocas=300)
    plt.plot(np.clip(h, None, 1e3), label=f'α = {alpha}')
plt.yscale('log'); plt.xlabel('época'); plt.ylabel('MSE (treino)')
plt.legend(); plt.show()

**Para pensar:** qual curva desce devagar? Qual oscila? Qual explode (diverge)?
O que acontece com `w` quando a taxa é grande demais?

## 5 · Desafio extra: regularização
Um polinômio de grau alto em `horas_estudo` pode decorar o treino. A regressão **Ridge** soma à perda a penalidade
$\lambda \sum w^2$ (no Scikit-learn, $\lambda$ se chama `alpha`). Compare pela validação cruzada.

In [ ]:
def poli(lam):
    passos = [('poli', PolynomialFeatures(degree=10)), ('padronizar', StandardScaler())]
    passos.append(('reg', LinearRegression() if lam == 0 else Ridge(alpha=lam)))
    return Pipeline(passos)

for lam in [0, 0.1, 10, 1000]:
    # TODO: MAE médio da validação cruzada (cv=5) para cada λ
    mae = -cross_val_score(poli(lam), x_tr.reshape(-1, 1), y_tr, cv=5,
                           scoring=___).mean()
    print(f'λ = {lam:>6}: MAE = {mae:.3f}')

## Para discutir
1. O modelo aprende com o **número de épocas** ou com a **atualização guiada pelo erro**?
2. O MAE de ~0,7 ponto é bom ou ruim? Depende de quê?
3. Por que a regressão múltipla erra menos que a reta só com horas?